# 航空与GOP锁定版本推理复核

这是**新增推理复核**，不会把新环境追认为旧实验环境。两份历史环境检查均未发现缓存或已加载模型；本程序用可追溯的新运行比较存档预测。

1. 在Google Colab上传打开本Notebook，建议在“运行时 → 更改运行时类型”选择T4 GPU。
2. 点击“全部运行”。出现上传窗口时，仅选择配套`07_Private_Replication_Inputs.zip`。
3. 等待两套推理完成，把自动下载的`Locked_Model_Replication_Results.zip`发给助手。无需改代码。

模型与tokenizer固定使用`3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7`。核心Python包使用明确版本，Torch固定2.11.0基本版本，实际CPU/CUDA构建和完整包清单另行记录。记录所有下载模型文件的SHA-256。

保留原调用中的文本处理、航空batch_size=32及默认截断长度、GOP batch_size=24及max_length=128。新运行采用直接模型调用、float32、eager attention和确定性算法设置，这些是明确的新复核设定，不声称完整还原旧运行。VADER阈值保持−0.05/0.05。

程序检查14,640条航空记录和4,483条GOP主分析记录，输出逐条新旧预测差异、组级负面比例、混淆计数、全部比较对及3/5/10 pp阈值下的反转变化。它不会运行预算实验或其他补充语料；若预测有变化，助手收到新输出后再复算相关下游分析。

输入ZIP包含第三方原始推文和账号字段，只供私有分析使用，勿上传公开仓库。输出不包含原始推文或账号字段，但private_row_outputs内含逐条预测/文本哈希，整个结果ZIP也应保持私有。程序在Colab本地推理，不调用远程文本推理API。

程序已完成输入对齐和统计比较路径的本地测试；尚未在此处执行神经网络推理。任何新结果都以本次Colab实际执行后生成的文件为准。


## 1 准备本次固定软件环境


In [ ]:
# 在新的专用Colab会话运行。软件安装后，推理由新的Python子进程执行。
import sys, subprocess, importlib.metadata, json
assert sys.version_info >= (3,10), '需要Python 3.10或更新版本'
pins = ['transformers==5.18.0','tokenizers==0.23.2','huggingface_hub==1.33.0',
        'vaderSentiment==3.3.2','numpy==2.1.3','pandas==2.2.3',
        'safetensors==0.8.0','scikit-learn==1.6.1']
try:
    torch_version = importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError:
    torch_version = ''
if torch_version.split('+')[0] != '2.11.0':
    pins.append('torch==2.11.0')
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*pins],check=True)
for item in pins:
    name,version=item.split('==')
    assert importlib.metadata.version(name).split('+')[0] == version, name+' 安装版本不符'
assert importlib.metadata.version('torch').split('+')[0]=='2.11.0'
print('软件准备完成。Torch的CPU/CUDA实际构建和完整环境会写入结果。')


## 2 上传配套私有输入包


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io
print('请选择 07_Private_Replication_Inputs.zip，只需上传这一份。')
uploaded = files.upload()
assert len(uploaded)==1, '请仅上传一份私有复核输入ZIP'
payload = next(iter(uploaded.values()))
assert zipfile.is_zipfile(io.BytesIO(payload)), '上传的不是ZIP文件'
INPUT_BUNDLE = Path('locked_replication_inputs.zip')
INPUT_BUNDLE.write_bytes(payload)
del payload, uploaded
print('输入包已接收；程序会先核验文件字节和逐行对应，再下载固定版本模型。')


## 3 执行锁定版本推理及新旧比较


In [ ]:
from pathlib import Path
import datetime, subprocess, sys
RESULT_READY = False
SCRIPT = '# SPDX-License-Identifier: MIT\n# Copyright (c) 2026 Beiyi Xiao\n"""New version-pinned inference comparison, not historical-version recovery."""\nimport argparse, datetime, hashlib, html, io, itertools, json, os, platform, re, subprocess, sys, unicodedata, zipfile\nfrom pathlib import Path\nimport importlib.metadata\nimport numpy as np\nimport pandas as pd\n\nMODEL_ID = \'cardiffnlp/twitter-roberta-base-sentiment-latest\'\nMODEL_REVISION = \'3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7\'\nEXPECTED_INPUT_HASHES = {\'Tweets.csv\': \'ea94b23f41892b290dec3330bb8cf9cb6b8bc669eaae5f3a84c40f7b0de8f15e\', \'airline_predictions.csv\': \'5674f3e56dc3667344ab43c867ee8dd4f61126823950524c3f5cd54d92f9eda2\', \'GOP_REL_ONLY.csv\': \'413bae11a4b04a414baffb63407788a3231a06b8ab806caaf9f930b5e2acec86\', \'gop_external_validation_results.zip\': \'8a93da7db63b6d2f5c0c2daec0faabdbab0b8cf7be415b1af30e3d49636968e7\'}\nLABELS = {\'negative\':0,\'neutral\':1,\'positive\':2}\n\ndef digest_bytes(b): return hashlib.sha256(b).hexdigest()\ndef file_hash(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\ndef norm(t):return \' \'.join(unicodedata.normalize(\'NFKC\',html.unescape(str(t))).casefold().split())\ndef text_hashes(s):return s.map(lambda t:digest_bytes(str(t).encode(\'utf-8\'))).to_numpy()\n\ndef load_inputs(bundle):\n    with zipfile.ZipFile(bundle) as z:\n        inputs={}\n        for name,expected in EXPECTED_INPUT_HASHES.items():\n            found=[n for n in z.namelist() if Path(n).name==name]\n            assert len(found)==1, f\'输入文件缺失或重名: {name}\'\n            b=z.read(found[0]);assert digest_bytes(b)==expected,f\'输入字节校验失败: {name}\'\n            inputs[name]=b\n    raw=pd.read_csv(io.BytesIO(inputs[\'Tweets.csv\']))\n    old=pd.read_csv(io.BytesIO(inputs[\'airline_predictions.csv\']))\n    assert len(raw)==len(old)==14640\n    assert np.array_equal(text_hashes(raw.text),old.text_sha256.to_numpy()),\'航空文本未逐行对齐\'\n    reference=raw.airline_sentiment.map(LABELS).to_numpy()\n    assert np.array_equal(reference,old.reference.to_numpy()),\'航空人工标签不一致\'\n    air=pd.DataFrame({\'group\':raw.airline,\'text\':raw.text.astype(str),\'source_position\':np.arange(len(raw)),\n        \'reference_class\':reference,\'human_negative\':(reference==0).astype(int),\n        \'old_vader_class\':old.vader,\'old_roberta_class\':old.roberta,\'text_sha256\':old.text_sha256})\n    for model,col in [(\'VADER\',\'vader\'),(\'RoBERTa\',\'roberta\')]:air[\'old_\'+model]=(old[col].to_numpy()==0).astype(int)\n\n    raw=pd.read_csv(io.BytesIO(inputs[\'GOP_REL_ONLY.csv\']))\n    assert len(raw)==13871\n    text=raw.text.fillna(\'\').astype(str).str.strip()\n    candidate=raw.candidate.fillna(\'\').astype(str).str.strip()\n    label=raw.sentiment.fillna(\'\').astype(str).str.strip().str.title()\n    excluded={\'no candidate mentioned\',\'none\',\'no candidate\',\'not mentioned\',\'not applicable\'}\n    keep=text.ne(\'\') & candidate.ne(\'\') & ~candidate.str.casefold().isin(excluded) & label.isin([\'Negative\',\'Neutral\',\'Positive\'])\n    valid=pd.DataFrame({\'text\':text[keep],\'group\':candidate[keep],\'sentiment\':label[keep],\'source_position\':np.flatnonzero(keep)})\n    valid[\'normalized\']=valid.text.map(norm)\n    first=valid.drop_duplicates(\'normalized\',keep=\'first\').reset_index(drop=True)\n    with zipfile.ZipFile(io.BytesIO(inputs[\'gop_external_validation_results.zip\'])) as z:\n        old=pd.read_csv(io.BytesIO(z.read(\'gop_external_validation_results/row_predictions_and_overlap_flags.csv\')))\n    assert len(valid)==6284 and len(first)==len(old)==4483\n    assert np.array_equal(text_hashes(first.text),old.text_sha256.to_numpy()),\'GOP文本未逐行对齐\'\n    assert np.array_equal(first.group.to_numpy(),old.candidate.to_numpy()),\'GOP分组不一致\'\n    assert np.array_equal(first.sentiment.to_numpy(),old.sentiment.to_numpy()),\'GOP人工标签不一致\'\n    first[\'reference_class\']=first.sentiment.str.lower().map(LABELS)\n    first[\'human_negative\']=(first.reference_class==0).astype(int)\n    first[\'text_sha256\']=old.text_sha256.to_numpy()\n    assert np.array_equal(first.human_negative.to_numpy(),old.human_negative.to_numpy())\n    for model in [\'VADER\',\'RoBERTa\']:first[\'old_\'+model]=old[model].to_numpy()\n    return air,first,old\n\ndef group_and_pair_reports(corpus,d):\n    group=[];pair=[];summary=[]\n    for model in [\'VADER\',\'RoBERTa\']:\n        for g,z in d.groupby(\'group\',sort=True):\n            p=z.human_negative.mean();rec={\'corpus\':corpus,\'model\':model,\'group\':g,\'n\':len(z),\'reference_negative_pct\':100*p}\n            for version in [\'old\',\'new\']:\n                y=z[version+\'_\'+model].to_numpy();gold=z.human_negative.to_numpy();q=y.mean()\n                tp=int(((gold==1)&(y==1)).sum());fn=int(((gold==1)&(y==0)).sum());fp=int(((gold==0)&(y==1)).sum());tn=int(((gold==0)&(y==0)).sum())\n                rec.update({version+\'_negative_pct\':100*q,version+\'_bias_pp\':100*(q-p),version+\'_TP\':tp,version+\'_FN\':fn,version+\'_FP\':fp,version+\'_TN\':tn})\n            rec[\'negative_share_change_pp\']=rec[\'new_negative_pct\']-rec[\'old_negative_pct\'];group.append(rec)\n        gm=pd.DataFrame([x for x in group if x[\'model\']==model]).set_index(\'group\')\n        model_pairs=[]\n        for a,b in itertools.combinations(gm.index,2):\n            gap=float(gm.loc[a,\'reference_negative_pct\']-gm.loc[b,\'reference_negative_pct\'])\n            rec={\'corpus\':corpus,\'model\':model,\'a\':a,\'b\':b,\'reference_gap_pp\':gap,\'reference_tie\':gap==0}\n            for version in [\'old\',\'new\']:\n                q=float(gm.loc[a,version+\'_negative_pct\']-gm.loc[b,version+\'_negative_pct\'])\n                rec.update({version+\'_gap_pp\':q,version+\'_strict_reversal\':int(gap*q<0),version+\'_predicted_tie\':q==0})\n            rec[\'reversal_status_changed\']=rec[\'old_strict_reversal\']!=rec[\'new_strict_reversal\'];model_pairs.append(rec);pair.append(rec)\n        model_pairs=pd.DataFrame(model_pairs)\n        old=d[\'old_\'+model].to_numpy();new=d[\'new_\'+model].to_numpy()\n        rec={\'corpus\':corpus,\'model\':model,\'n\':len(d),\'groups\':len(gm),\'pair_count\':len(model_pairs),\n             \'binary_label_changes\':int((old!=new).sum()),\'binary_label_agreement_pct\':100*float((old==new).mean()),\n             \'old_group_prevalence_MAE_pp\':float(gm.old_bias_pp.abs().mean()),\'new_group_prevalence_MAE_pp\':float(gm.new_bias_pp.abs().mean()),\n             \'maximum_absolute_group_share_change_pp\':float(gm.negative_share_change_pp.abs().max()),\n             \'changed_reversal_pairs\':int(model_pairs.reversal_status_changed.sum())}\n        for threshold in [0,3,5,10]:\n            subset=model_pairs[model_pairs.reference_gap_pp.abs()>=threshold]\n            rec[f\'gap{threshold}_pairs\']=len(subset)\n            for version in [\'old\',\'new\']:rec[f\'{version}_reversals_gap{threshold}\']=int(subset[version+\'_strict_reversal\'].sum())\n        if corpus==\'Airline\':\n            col=model.lower();rec[\'three_class_label_changes\']=int((d[\'old_\'+col+\'_class\']!=d[\'new_\'+col+\'_class\']).sum())\n        summary.append(rec)\n    return group,pair,summary\n\ndef save_comparison(air,gop,old_gop,out):\n    public=out/\'summary\';private=out/\'private_row_outputs\';public.mkdir(parents=True,exist_ok=True);private.mkdir(exist_ok=True)\n    groups=[];pairs=[];summaries=[]\n    for corpus,d in [(\'Airline\',air),(\'GOP\',gop)]:\n        g,p,s=group_and_pair_reports(corpus,d);groups+=g;pairs+=p;summaries+=s\n        keep=[c for c in d.columns if c not in {\'text\',\'normalized\',\'sentiment\'}]\n        d[keep].to_csv(private/(corpus.lower()+\'_row_comparison.csv\'),index=False)\n    pd.DataFrame(groups).to_csv(public/\'group_comparison.csv\',index=False)\n    pd.DataFrame(pairs).to_csv(public/\'pairwise_comparison.csv\',index=False)\n    table=pd.DataFrame(summaries);table.to_csv(public/\'comparison_summary.csv\',index=False)\n    # Canonical exports permit downstream budget reruns without changing old predictions.\n    pa=pd.DataFrame({\'reference\':air.reference_class,\'vader\':air.new_vader_class,\'roberta\':air.new_roberta_class,\n                     \'text_length\':air.text.str.len(),\'text_sha256\':air.text_sha256})\n    pa.to_csv(private/\'new_airline_predictions.csv\',index=False)\n    pg=old_gop.copy()\n    for m in [\'VADER\',\'RoBERTa\']:pg[m]=gop[\'new_\'+m].to_numpy()\n    pg.to_csv(private/\'new_gop_row_predictions_and_overlap_flags.csv\',index=False)\n    with zipfile.ZipFile(private/\'new_gop_external_validation_results.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        z.writestr(\'gop_external_validation_results/row_predictions_and_overlap_flags.csv\',pg.to_csv(index=False))\n        z.writestr(\'gop_external_validation_results/run_audit.json\',json.dumps({\'is_new_locked_inference\':True,\'model_id\':MODEL_ID,\'model_revision\':MODEL_REVISION,\'historical_revision_recovered\':False}))\n    return table\n\ndef main(bundle,out):\n    out.mkdir(parents=True,exist_ok=False)\n    air,gop,old_gop=load_inputs(bundle)\n    print(\'四个输入文件已校验；航空14,640行与GOP4,483行均逐行对齐。\',flush=True)\n    os.environ[\'CUBLAS_WORKSPACE_CONFIG\']=\':4096:8\'\n    import random,torch\n    from huggingface_hub import snapshot_download\n    from transformers import AutoTokenizer,AutoModelForSequenceClassification\n    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer\n    seed=20261017;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True)\n    if hasattr(torch.backends,\'cuda\'):torch.backends.cuda.matmul.allow_tf32=False\n    if hasattr(torch.backends,\'cudnn\'):torch.backends.cudnn.allow_tf32=False\n    device=torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\n    print(\'本次复核固定模型commit:\',MODEL_REVISION,\'；设备:\',device,flush=True)\n    if device.type==\'cpu\':print(\'未检测到GPU，CPU可以运行但速度可能明显更慢。\',flush=True)\n    # Download only files belonging to the exact commit; no automatic safetensors conversion PR.\n    snapshot=Path(snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,token=False,\n        allow_patterns=[\'config.json\',\'pytorch_model.bin\',\'vocab.json\',\'merges.txt\',\'special_tokens_map.json\',\'README.md\',\'.gitattributes\']))\n    assert snapshot.name==MODEL_REVISION,\'模型快照目录与固定commit不符\'\n    assets={p.name:{\'bytes\':p.stat().st_size,\'sha256\':file_hash(p)} for p in sorted(snapshot.iterdir()) if p.is_file()}\n    assert \'pytorch_model.bin\' in assets\n    tokenizer=AutoTokenizer.from_pretrained(str(snapshot),use_fast=True,local_files_only=True,trust_remote_code=False)\n    model=AutoModelForSequenceClassification.from_pretrained(str(snapshot),local_files_only=True,use_safetensors=False,\n        trust_remote_code=False,attn_implementation=\'eager\').float().to(device).eval()\n    id2label={int(k):str(v).lower() for k,v in model.config.id2label.items()}\n    assert set(id2label.values())==set(LABELS),\'非预期模型标签映射\'\n    analyzer=SentimentIntensityAnalyzer()\n    lex_path=Path(importlib.metadata.distribution(\'vaderSentiment\').locate_file(\'vaderSentiment/vader_lexicon.txt\'))\n    for corpus,d,batch,max_length in [(\'Airline\',air,32,None),(\'GOP\',gop,24,128)]:\n        compound=np.array([analyzer.polarity_scores(t)[\'compound\'] for t in d.text])\n        vc=np.where(compound<=-.05,0,np.where(compound>=.05,2,1));d[\'new_vader_class\']=vc;d[\'new_VADER\']=(vc==0).astype(int)\n        all_prob=[]\n        for i in range(0,len(d),batch):\n            kwargs={\'padding\':True,\'truncation\':True,\'return_tensors\':\'pt\'}\n            if max_length is not None:kwargs[\'max_length\']=max_length\n            encoded=tokenizer(d.text.iloc[i:i+batch].tolist(),**kwargs)\n            encoded={k:v.to(device) for k,v in encoded.items()}\n            with torch.inference_mode():prob=model(**encoded).logits.float().softmax(-1).cpu().numpy()\n            all_prob.append(prob)\n            if i==0 or (i//batch+1)%50==0 or i+batch>=len(d):print(corpus,min(i+batch,len(d)),\'/\',len(d),flush=True)\n        probs=np.concatenate(all_prob);rc=np.array([LABELS[id2label[int(j)]] for j in probs.argmax(axis=1)])\n        d[\'new_roberta_class\']=rc;d[\'new_RoBERTa\']=(rc==0).astype(int)\n        for j,label in id2label.items():d[\'new_roberta_probability_\'+label]=probs[:,j]\n    table=save_comparison(air,gop,old_gop,out)\n    versions={}\n    for name in [\'torch\',\'transformers\',\'tokenizers\',\'huggingface_hub\',\'vaderSentiment\',\'numpy\',\'pandas\',\'safetensors\',\'scikit-learn\']:\n        versions[name]=importlib.metadata.version(name)\n    runtime={\'status\':\'COMPLETE_NEW_LOCKED_INFERENCE\',\'recorded_utc\':datetime.datetime.now(datetime.timezone.utc).isoformat(),\n        \'historical_revision_recovered\':False,\'new_model_inference_executed\':True,\'model_id\':MODEL_ID,\'model_commit\':MODEL_REVISION,\n        \'model_files\':assets,\'source_input_sha256\':EXPECTED_INPUT_HASHES,\'python\':platform.python_version(),\'software_versions\':versions,\n        \'device\':str(device),\'gpu_name\':torch.cuda.get_device_name(0) if device.type==\'cuda\' else None,\'cuda_version\':torch.version.cuda,\n        \'cudnn_version\':torch.backends.cudnn.version(),\'seed\':seed,\'dtype\':\'float32\',\'attention\':\'eager\',\'deterministic_algorithms\':True,\n        \'tokenizer_class\':type(tokenizer).__name__,\'tokenizer_model_max_length\':int(tokenizer.model_max_length),\n        \'vader_lexicon_sha256\':file_hash(lex_path),\n        \'protocol\':{\'Airline\':{\'n\':14640,\'batch_size\':32,\'truncation\':True,\'max_length\':\'tokenizer default\'},\'GOP\':{\'n\':4483,\'batch_size\':24,\'truncation\':True,\'max_length\':128}},\n        \'text_preprocessing\':\'Airline original text; GOP strip then original normalization for deduplication only; no added @user/http replacement\',\n        \'code_sha256\':file_hash(__file__),\'budgets_and_auxiliary_corpora_rerun\':False,\n        \'boundary\':\'This new run tests stability of archived-output findings. It does not restore the historical inference environment.\'}\n    (out/\'summary/runtime_and_model_lock.json\').write_text(json.dumps(runtime,indent=2)+\'\\n\')\n    (out/\'private_row_outputs/environment_freeze.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\n    (out/\'private_row_outputs/requirements_observed.txt\').write_text(\'\\n\'.join(f\'{k}=={v}\' for k,v in versions.items())+\'\\n\')\n    report=[\'# New version-pinned inference comparison\',\'\',f\'Model commit: `{MODEL_REVISION}`. Historical model revision remains unavailable.\',\'\',\n        \'Both source corpora and archived prediction inputs passed exact-byte and row-alignment checks. This run re-executed VADER and RoBERTa on both primary analysis corpora. No new human labels or posts were collected.\',\'\']\n    for r in table.itertuples():\n        report.append(f\'- {r.corpus} / {r.model}: binary label changes {r.binary_label_changes}/{r.n}; strict reversals {r.old_reversals_gap0}/{r.pair_count} archived and {r.new_reversals_gap0}/{r.pair_count} new; changed reversal-pair identities {r.changed_reversal_pairs}; maximum group share change {r.maximum_absolute_group_share_change_pp:.6f} pp.\')\n    report+=[\'\',\'Budget and supplementary-corpus analyses have not been rerun by this notebook. If predictions differ, downstream analyses must be reassessed before updating the paper.\',\'\',\n        \'The summary folder contains aggregates and runtime/model provenance. The private_row_outputs folder contains individual prediction records and must not be placed in the public repository. No model weights, original tweet text or account fields are bundled in this result archive.\']\n    (out/\'summary/RESULT_README.md\').write_text(\'\\n\'.join(report)+\'\\n\')\n    manifest={str(p.relative_to(out)):file_hash(p) for p in sorted(out.rglob(\'*\')) if p.is_file()}\n    (out/\'RESULT_SHA256.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    archive=out.parent/\'Locked_Model_Replication_Results.zip\'\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.rglob(\'*\')):\n            if p.is_file():z.write(p,p.relative_to(out).as_posix())\n    print(\'\\n新复核完成。请下载并交回:\',archive,flush=True)\n    print(table[[\'corpus\',\'model\',\'binary_label_changes\',\'old_reversals_gap0\',\'new_reversals_gap0\',\'changed_reversal_pairs\']].to_string(index=False),flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--inputs\',required=True,type=Path);p.add_argument(\'--out\',required=True,type=Path);a=p.parse_args();main(a.inputs,a.out)\n'
RUNNER = Path('locked_model_replication.py')
RUNNER.write_text(SCRIPT,encoding='utf-8')
OUTPUT = Path('locked_replication_run_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
subprocess.run([sys.executable,str(RUNNER),'--inputs',str(INPUT_BUNDLE),'--out',str(OUTPUT)],check=True)
RESULT_READY = True


## 4 下载本次结果


In [ ]:
from google.colab import files
assert RESULT_READY, '本次推理尚未完整成功，请保留报错并发给助手，不能把旧结果当本次结果。'
print('请把下载的 Locked_Model_Replication_Results.zip 发给助手。')
print('结果含个体预测记录，暂勿将整个结果ZIP放入公开仓库。')
files.download('Locked_Model_Replication_Results.zip')
